# Vision Transformer 1D 对bacterial数据分类

使用Vision Transformer 1D(ViT1D)对bacteria中finetune数据集进行训练并预测test数据集。ViT1D是根据Vision Transformer文献中的原理及参考timm库包里的Vistion Transformer代码写的面向1D光谱数据的Transformer。由于输入的数据形状不是3D (C, H, W)形式而是(C, L)形式，因此在ViT1D里卷积核需要由2D卷积核换成1D卷积核，...。  

## 1 加载数据
直接使用'4.2_cnn/bacteria-1D'项目中的数据

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split

base_path = '../4.2_cnn/bacteria-1D/data'  # 4.2_cnn/bacteria-1D中数据位置(使用相对路径) 
# 载入finetune数据集
X_f = f'{base_path}/X_finetune.npy'
y_f = f'{base_path}/y_finetune.npy'
X_train = np.load(X_f)
y_train = np.load(y_f)

# 划分训练集和验证集
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# 载入独立测试集（训练/调参全程不接触，用于最终评估模型的泛化能力）
X_f = f'{base_path}/X_test.npy'
y_f = f'{base_path}/y_test.npy'
X_test = np.load(X_f)
y_test = np.load(y_f)

下面代码定义了一个继承于torch.utils.data.Dataset的数据集MyDataset，用来对原始数据进行tansform(转换)及reshape(变形)。

In [2]:
from torch.utils.data import Dataset
import torch

class MyDataset(Dataset):
    def __init__(self, features, labels, transform=None):
        """
        初始化数据集
        :param features: 包含数据的 np.ndarray
        :param labels: 包含标签的 np.ndarray
        :param transform: 用于转换数据的可选函数
        """
        self.transform = transform
        self.labels = torch.LongTensor(labels)
        self.features = features
        

    def __len__(self):
        """
        返回数据集中的样本数量
        """
        return len(self.features)

    def __getitem__(self, idx):
        """
        通过索引获取数据集中的样本
        """
        # 假设你的DataFrame中第一列是特征，第二列是标签
        feature = self.features[idx]
        label = self.labels[idx]

        feature = torch.FloatTensor(feature).reshape(1, -1)  # 增加出channel维

        # 返回特征和标签
        return feature, label


# 构建三个数据加载器：按batch_size分批取数；训练/验证集打乱样本顺序，测试集保持原序
batch_size = 32
train_loader = torch.utils.data.DataLoader(dataset=MyDataset(X_train, y_train), batch_size=batch_size, shuffle=True) 
valid_loader = torch.utils.data.DataLoader(dataset=MyDataset(X_valid, y_valid), batch_size=batch_size, shuffle=True) 
test_loader = torch.utils.data.DataLoader(dataset=MyDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

## 模型训练
1、外围框架函数  
下面函数来源于4.1_ann/mnist章节，直接copy过来的。

In [3]:
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import copy

def run(model, dataloader, device, epoch, criterion=nn.CrossEntropyLoss(), optimizer=None, training=False):   
    """跑完整个dataloader并统计（准确率, 平均损失）；training=True时额外执行反向传播与参数更新。"""
    total_size = 0
    total_loss = 0.0
    correct = 0

    for images, labels in tqdm(dataloader, desc=f"Epoch {epoch}", unit="batch"):
        images, labels = images.to(device), labels.to(device)        

        # 前向传播得到logits，再与标签计算交叉熵损失
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 仅训练阶段执行：清空上一步梯度 -> 反向传播计算梯度 -> 优化器更新参数
        if training:
            optimizer.zero_grad()     
            loss.backward()
            optimizer.step()
        
        # 按样本数加权累加损失，并累计样本总数与预测正确数，供循环结束后求平均
        total_loss += loss.item() * labels.size(0)
        total_size += labels.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        
    # 返回本轮的准确率与平均损失（损失按总样本数归一化）
    return correct/total_size, total_loss/total_size


def run_epoch(model, dataloader, device, epoch=0, criterion=nn.CrossEntropyLoss(), optimizer=None, training=False):
    """单个epoch的封装：训练时调model.train()，评估时调model.eval()并在no_grad下前向。"""
    acc, loss = 0, 0
    if training:
        # 训练模式（启用dropout等），调用run完成一轮训练
        model.train()
        acc, loss = run(model, dataloader, device, epoch, optimizer=optimizer, criterion=criterion, training=True)
    else:
        # 评估模式（关闭dropout），no_grad上下文不计算梯度：省显存且结果确定
        model.eval()
        with torch.no_grad():
            acc, loss = run(model, dataloader, device, epoch, optimizer=optimizer, criterion=criterion, training=False)

    return acc, loss


def train(model, device, train_loader, valid_loader, **kwargs):
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    
    # 训练模型
    num_epochs = 300  # 定义一个足够大的训练轮次
    num_patience = 10  # 定义耐心值，即验证集准确率不再提高的轮次
    patience_count = 0  # 记录验证集准确率不再提高的轮次
    best_accuracy = 0.0
    best_loss = 0.0
    best_model = None
    best_epoch = 0
    
    trains = []
    valids = []
    for epoch in range(num_epochs):
        train_acc, train_loss = run_epoch(model, train_loader, device, epoch, criterion=criterion, optimizer=optimizer, training=True)
        valid_acc, valid_loss = run_epoch(model, valid_loader, device, epoch, criterion=criterion, optimizer=optimizer, training=False)
    
        trains.append((train_acc, train_loss))
        valids.append((valid_acc, valid_loss))
    
        if epoch % 10 == 0:
            print(f"epoch:{epoch}. Train Accuracy: {train_acc:.4f}.  Train Loss: {train_loss:.6f}") 
            print(f"epoch:{epoch}. Valid Accuracy: {valid_acc:.4f}.  Valid Loss: {valid_loss:.6f}")
            
        if (valid_acc > best_accuracy) or (valid_acc == best_accuracy and valid_loss <  best_loss - 0.001):
            best_accuracy = valid_acc
            best_loss = valid_loss
            best_epoch = epoch
            best_model = copy.deepcopy(model.state_dict())
            patience_count = 0  # 重置耐心计数
        else:
            patience_count += 1
            print(f'Validating patience_count: {patience_count}')
        
        # 如果耐心计数达到耐心值，则跳出训练
        if patience_count >= num_patience:
            print("Early stopping!")
            print(f"epoch:{epoch}. Train Accuracy: {train_acc:.4f}.  Train Loss: {train_loss:.6f}") 
            print(f"epoch:{epoch}. Valid Accuracy: {valid_acc:.4f}.  Valid Loss: {valid_loss:.6f}")
            break
    return best_model, trains, valids

2、安装依赖包  
ViT1D(来源于文件ViT1D.py)中使用到einops。einops 是一个灵活且强大的张量操作库（具体功能请baidu）。

In [5]:
!pip install einops

3、模型训练

In [4]:
from ViT1D import ViT1D
import torch

# 自动选择计算设备：有可用CUDA的GPU时用cuda，否则回退到CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ViT超参数：类别数30（30类细菌）；每条光谱长度1000；每个patch长16；
# token嵌入维度48；Transformer编码器共6层(depth)；多头注意力共6个头
n_classes = 30
spectra_size = 1000
patch_size = 16
embed_dim = 48
depth=6
num_heads=6

# 按上述超参实例化1D Vision Transformer，并搬到计算设备上
model = ViT1D(spectra_size=spectra_size, patch_size=patch_size, num_classes=n_classes, embed_dim=embed_dim, depth=depth, num_heads=num_heads)
model = model.to(device)

In [7]:
# 启动训练（train内部含早停机制）：返回验证集表现最好的权重，以及每轮训练/验证的(准确率,损失)
best_model, trains, valids = train(model, device, train_loader, valid_loader)

Epoch 0: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 12.13batch/s]


epoch:0. Train Accuracy: 0.0604.  Train Loss: 3.425699
epoch:0. Valid Accuracy: 0.0917.  Valid Loss: 3.273952


Epoch 2: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 12.66batch/s]


Validating patience_count: 1


Epoch 4: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 12.24batch/s]


...

Validating patience_count: 8


Epoch 132: 100%|████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00,  9.76batch/s]


Validating patience_count: 9


Epoch 133: 100%|████████████████████████████████████████████████████████████████████| 19/19 [00:02<00:00,  9.33batch/s]

Validating patience_count: 10
Early stopping!


In [8]:
# 新建一个结构相同的空模型，载入验证集上的最佳权重，用于独立测试集评估
new_model = ViT1D(spectra_size=spectra_size, patch_size=patch_size, num_classes=n_classes, embed_dim=embed_dim, depth=depth, num_heads=num_heads)
new_model.load_state_dict(best_model)

# 在测试集上做纯前向评估（training=False），打印测试准确率与平均损失
test_acc, test_loss = run_epoch(new_model, test_loader, device, training=False)
print(f"Test Accuracy: {test_acc:.4f}.  Test Loss: {test_loss:.6f}") 

Epoch 0: 100%|██████████████████████████████████████████████████████████████████████| 94/94 [00:07<00:00, 11.84batch/s]

Test Accuracy: 0.6287.  Test Loss: 1.184794


## 思考作业
1、上面"模型训练"中"外围框架函数"直接从4.1_ann/mnist中copy过来。如果在原始处该代码**发生更改**(可能是修改bug或新增功能)，此处需不需要也更改？或者反之，此处发生修改，原始处是否应跟着修改？  
2、过了一段时间(10天后)，如果4.1_ann中的"外围框架函数"发生一个重要修改（希望其它处也应修改）。如何还记得4.4_transformer中也用到了该函数，也需要修改？特别是，如果4.1与4.4分是由两个不同工作者完成的工作，又该如何？**思考：如何让代码"Write Once, Run Anywhere"(WORA)?**
